# 第11单元 — 选择正确的统计检验（综合复习）

**机器翻译提示：本notebook文字由机器翻译生成，代码与数字结果和越南语原版完全一致，未经人工校对文字部分。**

第11单元专用notebook — 重新运行网页上已定义的6个真实研究情境，每个情境都使用决策树选出的确切检验方法。这里没有新的统计分析 -- 这是一个复习notebook，汇总了第05-10单元已学过的语法。

In [1]:
# 先运行这个单元格（Colab: 点击 ▶）。无需安装任何东西。
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """模拟的"240名学生班级"数据集 -- 与第03-08单元使用的
    相同 seed=2026，与仓库中的 lop_hoc_240.csv/.sav 完全一致
    （非真实数据，但100%可复现）。"""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Nam", "Nu"], n)
    method = rng.choice(["Truyen thong", "Du an"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Du an")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Du an") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("已创建 df:", df.shape)

已创建 df: (240, 14)


In [2]:
df["gain"] = df["posttest"] - df["pretest"]

## 情境1 — 比较3个独立学校，参数变量 → 单因素方差分析ANOVA（第06单元）

In [3]:
groups = [g["math"].values for _, g in df.groupby("school")]
f_stat, p_val = stats.f_oneway(*groups)
print(f"F={f_stat:.3f}, p={p_val:.4f}")

F=5.875, p=0.0032


## 情境2 — 同一对象，测量2次，参数变量 → 配对t检验（第05单元）

In [4]:
t, p = stats.ttest_rel(df["posttest"], df["pretest"])
print(f"t={t:.2f}, p={p:.2e}")

t=13.37, p=8.25e-31


## 情境3 — 3个独立组，顺序变量（李克特量表）→ Kruskal-Wallis检验（第07单元）

In [5]:
groups_h1 = [g["h1"].dropna().values for _, g in df.groupby("school")]
h_stat, p_kw = stats.kruskal(*groups_h1)
print(f"H={h_stat:.3f}, p={p_kw:.3f}")

H=2.844, p=0.241


## 情境4 — 先测量关联（Pearson r）再预测（回归）（第08+09单元）

In [6]:
mask = df["study_hours"] > 20
df.loc[mask, "study_hours"] = df.loc[mask, "study_hours"] / 10
r, p_r = stats.pearsonr(df["study_hours"], df["math"])
print(f"步骤1 -- Pearson r: r={r:.3f}, p={p_r:.2e}")

import statsmodels.api as sm
X = sm.add_constant(df[["study_hours"]])
m = sm.OLS(df["math"], X).fit()
print(f"步骤2 -- 回归: math = {m.params['const']:.2f} + {m.params['study_hours']:.2f}×study_hours, R²={m.rsquared:.3f}")

步骤1 -- Pearson r: r=0.582, p=4.07e-23


步骤2 -- 回归: math = 35.58 + 2.51×study_hours, R²=0.338


## 情境5 — 从多个变量中寻找潜在结构 → 因子分析（第10单元）

In [7]:
items = df[["h1","h2","h3","a1","a2","a3"]].dropna()
from factor_analyzer import FactorAnalyzer
fa = FactorAnalyzer(n_factors=2, rotation="varimax")
fa.fit(items)
print(pd.DataFrame(fa.loadings_, index=items.columns, columns=["因子1","因子2"]).round(3))

      因子1    因子2
h1 -0.043  0.747
h2 -0.003  0.603
h3 -0.096  0.670
a1  0.725 -0.092
a2  0.722 -0.083
a3  0.695 -0.016


## 情境6 — 将有相似特征的对象（学校）分组 → 聚类分析（第10单元）

In [8]:
RAW = "https://raw.githubusercontent.com/TatcataiTTN/for-Social-Science/main/SPSS/data/sav/"
truong = pd.read_csv(RAW + "vnm_truong_195_tong_hop.csv")
vars_ = ["EDULEAD","NEGSCLIM","STAFFSHORT","EDUSHORT","DIGPREP"]
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
d = truong[vars_].dropna()
X = StandardScaler().fit_transform(d)
km = KMeans(n_clusters=3, random_state=2026, n_init=10).fit(X)
print("每个簇的大小:", list(np.bincount(km.labels_)))

每个簇的大小: [np.int64(64), np.int64(49), np.int64(82)]


## 总结：6个情境快速查询表
| # | 研究问题 | 选定方法 | 单元 |
|---|---|---|---|
| 1 | 3个独立学校，数学成绩（参数）| 单因素方差分析ANOVA | 06 |
| 2 | 同一学生，前测→后测 | 配对t检验 | 05 |
| 3 | 3个独立学校，李克特评分 | Kruskal-Wallis检验 | 07 |
| 4 | 关联性与预测 | Pearson r + 回归 | 08, 09 |
| 5 | 6个题项的潜在结构 | 因子分析（EFA）| 10 |
| 6 | 195所学校按特征分组 | 聚类分析（K-means）| 10 |

这正是网页上展示的4问题决策树 -- 按顺序应用（目标 → 测量尺度 → 组数/关联方向）总能得到本表中正确的一行。